# 1D Velocity & Density Model Depth Profiles
Plots Vp/Vs, density, and shear modulus vs. depth for the DeShon (2006) 1D reference model.

In [ ]:
import sys
sys.path.insert(0, '/home/staff/chao/SSEinv/Nicoya/codes')
import utils_plot as utp

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

In [ ]:
savefig = True
# savefig = False

veldir = '/home/staff/chao/SSEinv/Nicoya/DeShon_2006GJI/'

# Coordinate system parameters (must match the FEniCS mesh)
lon0, lat0 = -84, 7       # reference point, from Christos's email
rot        = 45            # rotation angle in degrees (CCW)
x0, y0     = 130e3, 350e3 # offset in metres

# 1D velocity model
vel1d = pd.read_csv(veldir + 'DeShon2006_1Dmodel.csv', sep=r'\s+', skiprows=1,
                    names=['z', 'vp', 'vs', 'vp_vs_ratio'])
vel1d['z'] = vel1d['z'] * -1 * 1e3   # km → m, positive down → negative
vel1d = vel1d[vel1d['z'] <= 0].reset_index(drop=True)

# 1D density model
den1d = pd.read_csv(veldir + 'Density_1Dmodel.csv', sep=r'\s+', skiprows=1,
                    names=['z', 'den'])
den1d['z'] = den1d['z'] * -1 * 1e3   # km → m
den1d = den1d[den1d['z'] <= 0].reset_index(drop=True)
den1d['den_si'] = den1d['den'] * 1e3  # g/cm³ → kg/m³

# 3D velocity model — reproject lon/lat to local mesh coordinates (metres)
vel3d = pd.read_csv(veldir + 'DeShon2006_3Dmodel.csv', sep=',')
x_rot, y_rot = utp.LL2ckmd(vel3d['lon'], vel3d['lat'], lon0, lat0, rot)
vel3d['x'] = x_rot - x0  # metres, local mesh coordinate system
vel3d['y'] = y_rot - y0  # metres
vel3d['z'] = vel3d['z'] * -1 * 1e3   # km → m, positive down → negative
vel3d = vel3d[vel3d['z'] <= 0].reset_index(drop=True)

# Depth in km for plotting (positive = depth below surface)
depth_vel_km = -vel1d['z'] / 1e3
depth_den_km = -den1d['z'] / 1e3
depth_3d_km  = -vel3d['z'] / 1e3

# 1D shear modulus: μ = ρ × Vs² — direct merge on z (shared depth nodes, no interpolation)
vel1d_den  = vel1d.merge(den1d[['z', 'den_si']], on='z', how='left')
den_at_vel = vel1d_den['den_si'].values   # kept for print loop below
mu_gpa_1d  = vel1d_den['den_si'] * (vel1d['vs'] * 1e3)**2 / 1e9

# 3D shear modulus: direct merge on z (vel3d, den1d, vel1d all share same depth nodes)
# Also attach 1D reference values at each 3D point's depth (used for δln plots)
vel3d_ref = (vel3d
             .merge(den1d[['z', 'den_si']], on='z', how='left')
             .merge(vel1d[['z', 'vp', 'vs', 'vp_vs_ratio']].rename(
                        columns={'vp': 'vp_1d', 'vs': 'vs_1d', 'vp_vs_ratio': 'vpvs_1d'}),
                    on='z', how='left'))
assert vel3d_ref['den_si'].isna().sum() == 0, "Depth mismatch: den1d depths don't match vel3d"
assert vel3d_ref['vs_1d'].isna().sum()  == 0, "Depth mismatch: vel1d depths don't match vel3d"

mu_gpa_3d   = vel3d_ref['den_si'] * (vel3d_ref['vs']    * 1e3)**2 / 1e9  # 3D μ (GPa)
mu_1d_at_3d = vel3d_ref['den_si'] * (vel3d_ref['vs_1d'] * 1e3)**2 / 1e9  # 1D ref μ at same depths

print('Depth (km)  Vp    Vs   Vp/Vs  Density  Mu (GPa)')
for i in range(len(vel1d)):
    print(f'{depth_vel_km.iloc[i]:8.1f}   {vel1d.vp.iloc[i]:.2f}  {vel1d.vs.iloc[i]:.2f}  '
          f'{vel1d.vp_vs_ratio.iloc[i]:.3f}  {den_at_vel[i]:.2f}    {mu_gpa_1d.iloc[i]:.2f}')

print(f'\n3D model: {len(vel3d)} points at {vel3d["z"].nunique()} depth levels')
print(f'3D model x range: {vel3d["x"].min()/1e3:.1f} – {vel3d["x"].max()/1e3:.1f} km')
print(f'3D model y range: {vel3d["y"].min()/1e3:.1f} – {vel3d["y"].max()/1e3:.1f} km')
not700 = depth_3d_km < 700
print(f'3D shear modulus range (excl. 700 km): {mu_gpa_3d[not700].min():.1f} – {mu_gpa_3d[not700].max():.1f} GPa')
print(f'3D shear modulus range (incl. 700 km): {mu_gpa_3d.min():.1f} – {mu_gpa_3d.max():.1f} GPa')

In [ ]:
def layer_xy(depths_km, values):
    """Return (x, y) for a staircase plot of layered 1D model."""
    x, y = [], []
    for i in range(len(depths_km)):
        d_top = depths_km[i]
        d_bot = depths_km[i+1] if i+1 < len(depths_km) else depth_max
        x += [values[i], values[i]]
        y += [d_top, d_bot]
    return np.array(x), np.array(y)

In [ ]:
# ── Plotting ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(11, 5), sharey=True)
fig.subplots_adjust(wspace=0.08)

panel_labels = ["(a)", "(b)", "(c)", "(d)"]

depth_max = 100  # km

# ── Panel 1: Vp, Vs, and Vp/Vs — all on one bottom x-axis ───────────────────
ax = axes[0]

# 3D scatter
h1 = ax.scatter(vel3d['vp'],          depth_3d_km, color='steelblue', s=10, alpha=0.5,
                linewidths=0, zorder=1, label='Vp (3D)')
h2 = ax.scatter(vel3d['vs'],          depth_3d_km, color='salmon',    s=10, alpha=0.5,
                linewidths=0, zorder=1, label='Vs (3D)')
h3 = ax.scatter(vel3d['vp_vs_ratio'], depth_3d_km, color='gray',      s=10, alpha=0.5,
                linewidths=0, zorder=1, label='Vp/Vs (3D)')
# 1D staircases
xvp, yvp = layer_xy(depth_vel_km.values, vel1d['vp'].values)
xvs, yvs = layer_xy(depth_vel_km.values, vel1d['vs'].values)
xvr, yvr = layer_xy(depth_vel_km.values, vel1d['vp_vs_ratio'].values)
h4, = ax.plot(xvp, yvp, color='navy',    lw=2, zorder=2, label='Vp (1D)')
h5, = ax.plot(xvs, yvs, color='darkred', lw=2, zorder=2, label='Vs (1D)')
h6, = ax.plot(xvr, yvr, color='black',   lw=2, zorder=2, label='Vp/Vs (1D)')

ax.set_xlabel("Velocity (km/s) or Vp/Vs ratio", fontsize=11)
ax.set_ylabel('Depth (km)', fontsize=12)
ax.set_xlim(1, 10)
ax.set_ylim(depth_max, 0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(0.5))
ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(5))
ax.grid(True, which='major', ls='--', alpha=0.4)
legend_xy = (0.42, 0.35)   # axes coords: (0,0)=top-left, (1,1)=bottom-right (y-axis inverted)
# ax.legend(handles=[h4, h5, h6, h1, h2, h3], fontsize=9,
#           bbox_to_anchor=legend_xy, loc='upper left',
#           ncol=2, columnspacing=0.8, handlelength=1.2,
#           framealpha=0.9, edgecolor='0.7')
ax.legend(handles=[h4, h5, h6, h1, h2, h3], fontsize=9,
          bbox_to_anchor=legend_xy, loc='upper left',
          ncol=1, handlelength=1.2, framealpha=0.9, edgecolor='0.7')
ax.text(0.98, 0.92, panel_labels[0], transform=ax.transAxes, fontsize=14, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='right', va='bottom')


# ── Panel 2: Density ──────────────────────────────────────────────────────────
ax = axes[1]
xd, yd = layer_xy(depth_den_km.values, den1d['den'].values)
ax.plot(xd, yd, color='k', lw=2, label=r"$\rho$ (1D)")
ax.set_xlabel(r"Density $\rho$ (g/cm$^3$)", fontsize=11)
ax.set_xlim(2.5, 3.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(0.5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(0.1))
ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(5))
ax.legend(fontsize=9, loc='lower left')
ax.grid(True, which='major', ls='--', alpha=0.4)
ax.text(0.98, 0.92, panel_labels[1], transform=ax.transAxes, fontsize=14, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='right', va='bottom')

# ── Panel 3: Shear modulus from tabulated 3D model ───────────────────────────
ax = axes[2]
ax.scatter(mu_gpa_3d, depth_3d_km, color='gray', s=10, alpha=0.5,
           linewidths=0, zorder=1, label=r"$\mu$ (3D)")
xmu, ymu = layer_xy(depth_vel_km.values, mu_gpa_1d.values)
ax.plot(xmu, ymu, color='k', lw=2, zorder=2, label=r"$\mu$ (1D)")
ax.set_xlabel(r"Shear modulus $\mu$ (GPa)", fontsize=11)
ax.set_xlim(0, 120)
ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(5))
ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(5))
ax.legend(fontsize=9, loc='lower left')
ax.grid(True, which='major', ls='--', alpha=0.4)
# ax.set_title('Tabulated 3D model', fontsize=11)
ax.text(0.98, 0.92, panel_labels[2], transform=ax.transAxes, fontsize=14, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='right', va='bottom')

fig.suptitle('Reference model of DeShon et al. (2006)', fontsize=14, y=0.95)
plt.tight_layout()
if savefig:
    plt.savefig('../DeShon_2006GJI/ref_model_DeShon06.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# ── 2-Panel: δln of tabulated 3D model vs 1D reference ───────────────────────
# Left : δln(Vp)  and δln(Vp/Vs)  — quantities directly inverted in tomography
# Right: δln(Vs)  and δln(μ)      — derived quantities
#
# All δln computed from vel3d_ref (direct merge in cell-2, no interpolation):
#   vel3d_ref has columns: vp, vs, vp_vs_ratio (3D); vp_1d, vs_1d, vpvs_1d (1D ref)

dlnVp_tab   = (vel3d_ref['vp']          - vel3d_ref['vp_1d'])  / vel3d_ref['vp_1d']*100
dlnVpVs_tab = (vel3d_ref['vp_vs_ratio'] - vel3d_ref['vpvs_1d']) / vel3d_ref['vpvs_1d']*100
dlnVs_tab   = (vel3d_ref['vs']          - vel3d_ref['vs_1d'])  / vel3d_ref['vs_1d']*100
dlnmu_tab   = (mu_gpa_3d               - mu_1d_at_3d)          / mu_1d_at_3d*100

print(f"δln(Vp)    range: {dlnVp_tab.min():.3f}% – {dlnVp_tab.max():.3f}%")
print(f"δln(Vp/Vs) range: {dlnVpVs_tab.min():.3f}% – {dlnVpVs_tab.max():.3f}%")
print(f"δln(Vs)    range: {dlnVs_tab.min():.3f}% – {dlnVs_tab.max():.3f}%")
print(f"δln(μ)     range: {dlnmu_tab.min():.3f}% – {dlnmu_tab.max():.3f}%")

fig, axes = plt.subplots(1, 2, figsize=(7, 5), sharey=True)
fig.subplots_adjust(wspace=0.08)

depth_max = 100  # km

# Symmetric xlim: max abs across all 4 quantities
max_abs = max(dlnVp_tab.abs().max(), dlnVpVs_tab.abs().max(),
              dlnVs_tab.abs().max(), dlnmu_tab.abs().max())
xlim = (-max_abs * 1.08, max_abs * 1.08)

# ── Left panel: Vp and Vp/Vs ─────────────────────────────────────────────────
ax = axes[0]
ax.scatter(dlnVp_tab,   depth_3d_km, color='steelblue', s=8, alpha=0.5,
           linewidths=0, zorder=1, label='δln(Vp)')
ax.scatter(dlnVpVs_tab, depth_3d_km, color='gray',      s=8, alpha=0.5, marker='D',
           linewidths=0, zorder=2, label='δln(Vp/Vs)')
ax.axvline(0, color='k', lw=1.5, zorder=3)
ax.set_xlabel('δln = (3D − 1D) / 1D', fontsize=11)
ax.set_ylabel('Depth (km)', fontsize=12)
ax.set_xlim(xlim)
ax.set_ylim(depth_max, 0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(0.2*100))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(0.05*100))
ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(5))
ax.legend(fontsize=10, loc='lower left', markerscale=2)
ax.grid(True, which='major', ls='--', alpha=0.4)
ax.set_title('Vp  &  Vp/Vs (inverted)', fontsize=11)
ax.text(0.02, 0.97, '(a)', transform=ax.transAxes, fontsize=12,
        fontweight='bold', va='top', ha='left')

# ── Right panel: Vs and μ ──────────────────────────────────────────────────────
ax = axes[1]
ax.scatter(dlnVs_tab,  depth_3d_km, color='salmon', s=8, alpha=0.5,
           linewidths=0, zorder=2, label='δln(Vs)')
ax.scatter(dlnmu_tab,  depth_3d_km, color='gray', s=8, alpha=0.5, marker='D',
           linewidths=0, zorder=1, label='δln(μ)')
ax.axvline(0, color='k', lw=1.5, zorder=3)
ax.set_xlabel('δln = (3D − 1D) / 1D', fontsize=11)
ax.set_xlim(xlim)
ax.xaxis.set_major_locator(ticker.MultipleLocator(0.2*100))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(0.05*100))
ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(5))
ax.legend(fontsize=10, loc='lower left', markerscale=2)
ax.grid(True, which='major', ls='--', alpha=0.4)
ax.set_title('Vs  &  μ (derived)', fontsize=11)
ax.text(0.02, 0.97, '(b)', transform=ax.transAxes, fontsize=12,
        fontweight='bold', va='top', ha='left')

fig.suptitle('Anomaly δln of tabulated 3D model relative to 1D reference\n(DeShon et al. 2006)', fontsize=12)
plt.tight_layout()
if savefig:
    plt.savefig('../DeShon_2006GJI/plt_dlnmu_tabulated.pdf', bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
import h5py

# ── Load mesh-projected CG2 shear modulus from _dofs.h5 ──────────────────────
# Generated by test_higher_CGmu_synth_stripeslip_3DCK_noi.py via save_function_with_dofs()
# Contains ALL CG2 DOF coordinates (mesh vertices + edge midpoints) and μ values (GPa)
# Confirmed byte-for-byte identical to:
dofs_file = '/home/staff/chao/SSEinv/Nicoya/rst_locking/mu_true_nicoyaCKden_une_all_DeShon3D_ref_1_hull_CG2_dofs.h5'
# dofs_file_scaled = '/home/staff/chao/SSEinv/Nicoya/syn_slip/mu_true_nicoyaCKden_une_sm_DeShon3D_ref_4_hull_CG2_dofs.h5'

# dofs_file        = '/home/staff/chao/SSEinv/Nicoya/syn_slip/mu_true_test_orinicoyaCKden_une_sm_DeShon3D_ref_4_hull_CG2_dofs.h5'
# dofs_file_scaled = '/home/staff/chao/SSEinv/Nicoya/syn_slip/mu_true_test1_nicoyaCKden_une_sm_DeShon3D_ref_4_hull_CG2_dofs.h5'
# dofs_file_scaled = '/home/staff/chao/SSEinv/Nicoya/syn_slip/mu_true_testfact2_nicoyaCKden_une_sm_DeShon3D_ref_4_hull_CG2_dofs.h5'
dofs_file_scaled = '/home/staff/chao/SSEinv/Nicoya/syn_slip/mu_true_testfact2.5_nicoyaCKden_une_sm_DeShon3D_ref_4_hull_CG2_dofs.h5'

# #####
# ## for testing, if building 3D model from 1D interp rather than layer lookup, and amplifying makes a difference
# dofs_file = '/home/staff/chao/SSEinv/Nicoya/rst_locking/mu_true_nicoyaCKden_une_all_DeShon3D_ref_1_hull_test1_CG2_dofs.h5'
# dofs_file_scaled = '/home/staff/chao/SSEinv/Nicoya/rst_locking/mu_true_nicoyaCKden_une_all_DeShon3D_ref_2_hull_test1_CG2_dofs.h5'
# #####

dofs_file_vs     = '/home/staff/chao/SSEinv/Nicoya/rst_locking/vs_true_nicoyaCKden_une_all_DeShon3D_ref_1_hull_CG2_dofs.h5'

with h5py.File(dofs_file, 'r') as f:
    dof_coords    = f['coordinates'][:]   # (N_dofs, 3) metres
    dof_mu_gpa    = f['values'][:]        # shear modulus in GPa
    print(f"Unscaled μ — Loaded: '{f.attrs['attr_name']}'  CG{f.attrs['cg_degree']}")
    print(f"  DOFs: {f.attrs['n_dofs']:,}  (mesh vertices: {f.attrs['n_vertices']:,},"
          f"  ratio: {f.attrs['n_dofs']/f.attrs['n_vertices']:.1f}x)")

with h5py.File(dofs_file_scaled, 'r') as f:
    dof_mu_gpa_scaled = f['values'][:]
    print(f"Scaled   μ — Loaded: '{f.attrs['attr_name']}'  CG{f.attrs['cg_degree']}")
    print(f"  DOFs: {f.attrs['n_dofs']:,}  (mesh vertices: {f.attrs['n_vertices']:,},"
          f"  ratio: {f.attrs['n_dofs']/f.attrs['n_vertices']:.1f}x)")

with h5py.File(dofs_file_vs, 'r') as f:
    dof_vs_kms = f['values'][:]           # Vs in km/s
    print(f"Vs         — Loaded: '{f.attrs['attr_name']}'  CG{f.attrs['cg_degree']}")
    print(f"  DOFs: {f.attrs['n_dofs']:,}  (mesh vertices: {f.attrs['n_vertices']:,},"
          f"  ratio: {f.attrs['n_dofs']/f.attrs['n_vertices']:.1f}x)")

# z is negative downward in mesh coords; convert to positive depth in km
depth_dof_km = -dof_coords[:, 2] / 1e3

print(f"\nDepth range     : {depth_dof_km.min():.1f} - {depth_dof_km.max():.1f} km")
print(f"Unscaled μ range: {dof_mu_gpa.min():.2f} - {dof_mu_gpa.max():.2f} GPa")
print(f"Scaled   μ range: {dof_mu_gpa_scaled.min():.2f} - {dof_mu_gpa_scaled.max():.2f} GPa")
print(f"Vs range        : {dof_vs_kms.min():.2f} - {dof_vs_kms.max():.2f} km/s")

In [ ]:
from scipy.spatial import Delaunay

# ── Classify mesh DOFs as inside 3D hull vs 1D background ────────────────────
# Mirrors exactly utils.process_velocity_models_hull (Steps 3 & 7):
#   data_points = vel3d_clean[['x', 'y', 'z']].values
#   hull = Delaunay(data_points)
#   inside = hull.find_simplex(V_coords) >= 0
# vel3d['x'], vel3d['y'] are in metres (reprojected via LL2ckmd in cell-2)
# vel3d['z'] is in metres; dof_coords are in metres → same coordinate system

hull_3d = Delaunay(vel3d[['x', 'y', 'z']].values)

# find_simplex >= 0 → inside hull (3D region); -1 → outside (1D background)
inside_3d = hull_3d.find_simplex(dof_coords) >= 0

print(f"DOFs inside 3D model hull : {inside_3d.sum():,}  ({100*inside_3d.mean():.1f}%)")
print(f"DOFs in 1D background     : {(~inside_3d).sum():,}  ({100*(~inside_3d).mean():.1f}%)")
print()
print(f"Depth range (3D hull DOFs) : {depth_dof_km[inside_3d].min():.1f} – {depth_dof_km[inside_3d].max():.1f} km")
print(f"Depth range (1D bkg DOFs)  : {depth_dof_km[~inside_3d].min():.1f} – {depth_dof_km[~inside_3d].max():.1f} km")
print()
# Histogram of 1D background DOFs by depth bracket
import numpy as np
brackets = [(0, 10), (10, 30), (30, 60), (60, 90), (90, 150), (150, 250), (250, 400)]
d_out = depth_dof_km[~inside_3d]
print("1D background DOF count by depth bracket:")
for lo, hi in brackets:
    n = np.sum((d_out >= lo) & (d_out < hi))
    print(f"  {lo:3d}–{hi:3d} km : {n:,}")

In [ ]:
# ── Verify: outside hull, scaled and unscaled should be identical ─────────────
diff_outside = dof_mu_gpa_scaled[~inside_3d] - dof_mu_gpa[~inside_3d]
max_diff = np.abs(diff_outside).max()
print(f"Outside hull DOFs : {(~inside_3d).sum():,}")
print(f"Max |scaled - unscaled| : {max_diff:.2e} GPa")
if max_diff < 1e-10:
    print("✓ PASSED: 1D background region is identical in scaled and unscaled models")
else:
    print("✗ WARNING: Differences detected in 1D background region")

In [ ]:
# ── Single-Panel: Mesh-projected shear modulus (CG2 DOFs) ────────────────────
fig, ax = plt.subplots(1, 1, figsize=(5, 6))

depth_max = 100  # km

xmu, ymu = layer_xy(depth_vel_km.values, mu_gpa_1d.values)

ax.scatter(dof_mu_gpa[inside_3d], depth_dof_km[inside_3d], color='tomato', s=2, alpha=0.5,
           linewidths=0, zorder=1, label='μ (3D)')
ax.plot(xmu, ymu, color='k', lw=2, zorder=2, label='μ (1D)')
ax.set_xlabel('Shear modulus μ (GPa)', fontsize=12)
ax.set_ylabel('Depth (km)', fontsize=12)
ax.set_xlim(0, 100)
ax.set_ylim(depth_max, 0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(5))
ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(5))
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, which='major', ls='--', alpha=0.4)
ax.set_title('Mesh-projected shear modulus (hull region of 3D, CG2, unscaled)', fontsize=12)

plt.tight_layout()

if savefig:
    plt.savefig('../DeShon_2006GJI/plt_mu_mesh_projected.pdf', bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
# ── 2-Panel: Unscaled (left) vs Scaled (right) — 3D-hull DOFs only ───────────
fig, axes = plt.subplots(1, 2, figsize=(9, 6), sharey=True)
fig.subplots_adjust(wspace=0.08)

depth_max = 100  # km
xmu, ymu = layer_xy(depth_vel_km.values, mu_gpa_1d.values)

# ── Panel 1: Unscaled ─────────────────────────────────────────────────────────
ax = axes[0]
ax.scatter(dof_mu_gpa[inside_3d], depth_dof_km[inside_3d],
           color='tomato', s=2, alpha=0.5, linewidths=0, zorder=1,
           label='μ (unscaled, 3D hull)')
ax.plot(xmu, ymu, color='k', lw=2, zorder=2, label='μ (1D reference)')
ax.set_xlabel('Shear modulus μ (GPa)', fontsize=12)
ax.set_ylabel('Depth (km)', fontsize=12)
ax.set_ylim(depth_max, 0)
ax.set_xlim(0, 160)
ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(5))
# ax.set_xscale('log')
# ax.set_xlim(5, 200)
# ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:g}'))
ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(5))
ax.legend(fontsize=9, loc='upper right', markerscale=3)
ax.grid(True, which='major', ls='--', alpha=0.4)
ax.set_title('Unscaled', fontsize=11)

# ── Panel 2: Scaled ───────────────────────────────────────────────────────────
ax = axes[1]
ax.scatter(dof_mu_gpa_scaled[inside_3d], depth_dof_km[inside_3d],
           color='tomato', s=2, alpha=0.5, linewidths=0, zorder=1,
           label='μ (scaled, 3D hull)')
ax.plot(xmu, ymu, color='k', lw=2, zorder=2, label='μ (1D reference)')
ax.set_xlabel('Shear modulus μ (GPa)', fontsize=12)
ax.set_xlim(0, 160)
ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(5))
# ax.set_xscale('log')
# ax.set_xlim(5, 200)
# ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:g}'))
ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(5))
ax.legend(fontsize=9, loc='upper right', markerscale=3)
ax.grid(True, which='major', ls='--', alpha=0.4)
ax.set_title('Scaled', fontsize=11)

fig.suptitle('Mesh-projected shear modulus — 3D-hull DOFs only (CG2)', fontsize=13)
plt.tight_layout()
if savefig:
    plt.savefig('../DeShon_2006GJI/plt_mu_scaled_vs_unscaled.pdf', bbox_inches='tight', dpi=150)
plt.show()



In [ ]:
# ── 1D reference method: match how the μ model was built ─────────────────────
# 'step'   → consistent with process_velocity_models_hull  + scale_shear_modulus_by_1d
# 'interp' → consistent with process_velocity_models_hull_interp + scale_shear_modulus_by_1d_interp
mu_1d_ref_method = 'step'   # <-- change here
# mu_1d_ref_method = 'interp'   # <-- change here

vel1d_s = vel1d.dropna(subset=['z', 'vs']).sort_values('z', ascending=False).reset_index(drop=True)
den1d_s = den1d.dropna(subset=['z']).sort_values('z', ascending=False).reset_index(drop=True)

neg_z     = -dof_coords[:, 2]          # positive depths (metres)
neg_z_vel = -vel1d_s['z'].values       # ascending
neg_z_den = -den1d_s['z'].values

if mu_1d_ref_method == 'step':
    # Step-function lookup (constant within each layer; deeper layer value at boundaries)
    idx_vs  = np.clip(np.searchsorted(neg_z_vel, neg_z, side='right') - 1, 0, len(vel1d_s)  - 1)
    idx_den = np.clip(np.searchsorted(neg_z_den, neg_z, side='right') - 1, 0, len(den1d_s) - 1)
    vs_ref_at_dofs  = vel1d_s['vs'].values[idx_vs]          # km/s
    den_ref_at_dofs = den1d_s['den_si'].values[idx_den]     # kg/m3
else:  # 'interp'
    # Linear interpolation between tabulated nodes (consistent with get_layered_1d_value_interp)
    vs_ref_at_dofs  = np.interp(neg_z, neg_z_vel, vel1d_s['vs'].values)       # km/s
    den_ref_at_dofs = np.interp(neg_z, neg_z_den, den1d_s['den_si'].values)   # kg/m3

mu_ref_at_dofs = den_ref_at_dofs * (vs_ref_at_dofs * 1e3)**2 / 1e9  # GPa

# Anomaly in %, dln(mu) = (mu - mu_ref(z)) / mu_ref(z)
dlnmu_orig   = (dof_mu_gpa        - mu_ref_at_dofs) / mu_ref_at_dofs *100
dlnmu_scaled = (dof_mu_gpa_scaled - mu_ref_at_dofs) / mu_ref_at_dofs *100

print(f'1D ref method        : {mu_1d_ref_method}')
print(f'mu_ref range         : {mu_ref_at_dofs.min():.2f} - {mu_ref_at_dofs.max():.2f} GPa')
print(f'dln(mu) unscaled     : {dlnmu_orig.min():.3f}% - {dlnmu_orig.max():.3f}%')
print(f'dln(mu) scaled       : {dlnmu_scaled.min():.3f}% - {dlnmu_scaled.max():.3f}%')
print(f'  (3D-hull only)')
print(f'dln(mu) unscaled hull: {dlnmu_orig[inside_3d].min():.3f}% - {dlnmu_orig[inside_3d].max():.3f}%')
print(f'dln(mu) scaled hull  : {dlnmu_scaled[inside_3d].min():.3f}% - {dlnmu_scaled[inside_3d].max():.3f}%')


In [ ]:
# ── 2-Panel: Tabulated vs mesh-projected δln(Vs) and δln(μ) ──────────────────
# Checks that the mesh interpolation correctly captures the 3D velocity anomalies.
# Tabulated: values at vel3d grid nodes (input to process_velocity_models_hull)
# Mesh-projected: CG2 DOFs inside hull (output after FEniCS interpolation)

dlnVs_dof  = (dof_vs_kms - vs_ref_at_dofs) / vs_ref_at_dofs *100

not700_tab = depth_3d_km < 700   # exclude 700 km boundary node (identical to 1D → δln=0)

# Symmetric xlim: max abs across both tabulated and mesh-projected (excl. 700 km)
max_abs = max(np.abs(dlnVs_tab[not700_tab]).max(), np.abs(dlnmu_tab[not700_tab]).max(),
              np.abs(dlnVs_dof[inside_3d]).max(),  np.abs(dlnmu_orig[inside_3d]).max())
xlim = (-max_abs * 1.08, max_abs * 1.08)

fig, axes = plt.subplots(1, 2, figsize=(8, 6), sharey=True)
fig.subplots_adjust(wspace=0.08)

depth_max = 100  # km

# ── Left panel: δln(Vs) ───────────────────────────────────────────────────────
ax = axes[0]
ax.scatter(dlnVs_tab[not700_tab],  depth_3d_km[not700_tab],
           color='steelblue', s=20, alpha=0.6, linewidths=0, zorder=1, label='Tabulated 3D')
ax.scatter(dlnVs_dof[inside_3d],   depth_dof_km[inside_3d],
           color='tomato',    s=2,  alpha=0.5, linewidths=0, zorder=2, label='Mesh-projected (CG2)')
ax.axvline(0, color='k', lw=1.5, zorder=3)
ax.set_xlabel('δln(Vs) = (Vs − Vs_ref) / Vs_ref (%)', fontsize=11)
ax.set_ylabel('Depth (km)', fontsize=12)
ax.set_xlim(xlim)
ax.set_ylim(depth_max, 0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(0.2*100))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(0.05*100))
ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(5))
ax.legend(fontsize=10, loc='lower left', markerscale=3)
ax.grid(True, which='major', ls='--', alpha=0.4)
ax.set_title('δln(Vs)', fontsize=11)
ax.text(0.02, 0.97, '(a)', transform=ax.transAxes, fontsize=12,
        fontweight='bold', va='top', ha='left')

# ── Right panel: δln(μ) ───────────────────────────────────────────────────────
ax = axes[1]
ax.scatter(dlnmu_tab[not700_tab],  depth_3d_km[not700_tab],
           color='gray',   s=20, alpha=0.6, linewidths=0, zorder=1, label='Tabulated 3D')
ax.scatter(dlnmu_orig[inside_3d],  depth_dof_km[inside_3d],
           color='tomato', s=2,  alpha=0.5, linewidths=0, zorder=2, label='Mesh-projected (CG2)')
ax.axvline(0, color='k', lw=1.5, zorder=3)
ax.set_xlabel('δln(μ) = (μ − μ_ref) / μ_ref (%)', fontsize=11)
ax.set_xlim(xlim)
ax.xaxis.set_major_locator(ticker.MultipleLocator(0.2*100))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(0.05*100))
ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(5))
ax.legend(fontsize=10, loc='lower left', markerscale=3)
ax.grid(True, which='major', ls='--', alpha=0.4)
ax.set_title('δln(μ)', fontsize=11)
ax.text(0.02, 0.97, '(b)', transform=ax.transAxes, fontsize=12,
        fontweight='bold', va='top', ha='left')

fig.suptitle('Tabulated 3D vs mesh-projected δln  (hull region, CG2, unscaled)', fontsize=12)
plt.tight_layout()

if savefig:
    plt.savefig('../DeShon_2006GJI/plt_dln_tabulated_vs_meshprojected.pdf', bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
# ── 2-Panel: δln(μ) unscaled (left) vs scaled (right) — 3D-hull DOFs only ────
fig, axes = plt.subplots(1, 2, figsize=(9, 6), sharey=True)
fig.subplots_adjust(wspace=0.08)

depth_max = 100  # km

# Symmetric xlim: use max abs across both panels
max_abs = max(np.abs(dlnmu_orig[inside_3d]).max(), np.abs(dlnmu_scaled[inside_3d]).max())
xlim = (-max_abs * 1.05, max_abs * 1.05)

for ax, dlnmu, title, label in zip(
        axes,
        [dlnmu_orig, dlnmu_scaled],
        # [dlnmu_orig, dlnmu_orig*4],
        ['Unscaled', 'Scaled'],
        ['δln(μ) (unscaled, 3D hull)', 'δln(μ) (scaled, 3D hull)']):

    ax.scatter(dlnmu[inside_3d], depth_dof_km[inside_3d],
               color='tomato', s=2, alpha=0.5, linewidths=0, zorder=1, label=label)
    ax.axvline(0, color='k', lw=2, zorder=2, label='μ_ref(z)  [δln=0]')
    ax.set_xlabel('δln(μ) = (μ − μ_ref) / μ_ref (%)', fontsize=11)
    ax.set_xlim(xlim)
    ax.set_ylim(depth_max, 0)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(0.5*100))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(0.1*100))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
    ax.yaxis.set_minor_locator(ticker.MultipleLocator(5))
    ax.legend(fontsize=9, loc='lower left', markerscale=3)
    ax.grid(True, which='major', ls='--', alpha=0.4)
    ax.set_title(title, fontsize=11)

axes[0].set_ylabel('Depth (km)', fontsize=12)

fig.suptitle('Shear modulus anomaly δln(μ) — 3D-hull DOFs only (CG2)', fontsize=13)
plt.tight_layout()

# if savefig:
#     plt.savefig('../DeShon_2006GJI/plt_dlnmu_scaled_vs_unscaled.pdf', bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
# ── 4-Panel 2×2 Plot: velocity, density, tabulated μ, mesh-projected μ ────────
fig, axes = plt.subplots(2, 2, figsize=(8, 10), sharey=True)
fig.subplots_adjust(wspace=0.08)
axes = axes.flatten()

panel_labels = ["(a)", "(b)", "(c)", "(d)"]

depth_max = 100  # km

xmu, ymu = layer_xy(depth_vel_km.values, mu_gpa_1d.values)

# ── Panel 1: Original ─────────────────────────────────────────────────────────
ax = axes[0]
ax.scatter(dof_mu_gpa[inside_3d], depth_dof_km[inside_3d],
           color='gray', s=2, alpha=0.5, linewidths=0, zorder=1,
           label=r"$\mu$ (3D)")
ax.plot(xmu, ymu, color='k', lw=2, zorder=2, label=r"$\mu$ (1D ref.)")
ax.set_xlabel(r"Shear modulus $\mu$ (GPa)", fontsize=11)
ax.set_ylabel('Depth (km)', fontsize=11)
ax.set_ylim(depth_max, 0)
ax.set_xlim(0, 120)
ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(5))
# ax.set_xscale('log')
# ax.set_xlim(5, 200)
# ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:g}'))
ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(5))
ax.legend(fontsize=9, loc='lower left', markerscale=3)
ax.grid(True, which='major', ls='--', alpha=0.4)
ax.set_title('Original', fontsize=11)
ax.text(0.05, 0.25, 
        f"{round(dof_mu_gpa.min()):d}-{round(dof_mu_gpa.max()):d} GPa", 
        transform=ax.transAxes, fontsize=15)
ax.text(0.98, 0.92, panel_labels[0], transform=ax.transAxes, fontsize=14, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='right', va='bottom')

# ── Panel 2: Amplified ───────────────────────────────────────────────────────────
ax = axes[2]
ax.scatter(dof_mu_gpa_scaled[inside_3d], depth_dof_km[inside_3d],
           color='gray', s=2, alpha=0.5, linewidths=0, zorder=1,
           label=r"$\mu$ (3D)")
ax.plot(xmu, ymu, color='k', lw=2, zorder=2, label=r"$\mu$ (1D ref.)")
ax.set_xlabel(r"Shear modulus $\mu$ (GPa)", fontsize=11)
ax.set_ylabel('Depth (km)', fontsize=11)
ax.set_xlim(0, 120)
ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(5))
# ax.set_xscale('log')
# ax.set_xlim(5, 200)
# ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:g}'))
ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(5))
ax.legend(fontsize=9, loc='lower left', markerscale=3)
ax.grid(True, which='major', ls='--', alpha=0.4)
ax.set_title('Amplified by a factor of 2.5', fontsize=11)
ax.text(0.05, 0.25, 
        f"{round(dof_mu_gpa_scaled.min()):d}-{round(dof_mu_gpa_scaled.max()):d} GPa", 
        transform=ax.transAxes, fontsize=15)
ax.text(0.98, 0.92, panel_labels[2], transform=ax.transAxes, fontsize=14, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='right', va='bottom')


# Symmetric xlim: use max abs across both panels
max_abs = max(np.abs(dlnmu_orig[inside_3d]).max(), np.abs(dlnmu_scaled[inside_3d]).max())
# xlim = (-max_abs * 1.05, max_abs * 1.05)
xlim = (-1.5*100, 1.5*100)

for ax, dlnmu, title, label in zip(
        [axes[1], axes[3]],
        [dlnmu_orig, dlnmu_scaled],
        ['Original', 'Amplified by a factor of 2.5'],
        [r"dln($\mu$) (3D)", r"dln($\mu$) (3D)"]):

    ax.scatter(dlnmu[inside_3d], depth_dof_km[inside_3d],
               color='gray', s=2, alpha=0.5, linewidths=0, zorder=1, label=label)
    ax.axvline(0, color='k', lw=2, zorder=2, label=r"dln($\mu$) (1D ref.)")
    ax.set_xlabel(r"dln($\mu$) = $(\mu - \mu_{ref}) / \mu_{ref}$ (%)", fontsize=11)
    ax.set_xlim(xlim)
    ax.set_ylim(depth_max, 0)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(0.5*100))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(0.1*100))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
    ax.yaxis.set_minor_locator(ticker.MultipleLocator(5))
    ax.legend(fontsize=9, loc='lower left', markerscale=3)
    ax.grid(True, which='major', ls='--', alpha=0.4)
    ax.set_title(title, fontsize=11)

# axes[3].set_xlabel(r"dln($\mu$) = $(\mu - \mu_{ref}) / \mu_{ref}$ (%)", fontsize=11)
axes[1].text(0.98, 0.92, panel_labels[1], transform=axes[1].transAxes, fontsize=14, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='right', va='bottom')
axes[3].text(0.98, 0.92, panel_labels[3], transform=axes[3].transAxes, fontsize=14, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='right', va='bottom')

fig.suptitle(r"Mesh-projected shear modulus $\mu$", fontsize=13)
plt.tight_layout()
if savefig:
    plt.savefig('../DeShon_2006GJI/mu_original_vs_amplified.png', bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
# ── Linear-interpolation 1D reference μ ────────────────────────────────────
# neg_z, neg_z_vel, neg_z_den, vel1d_s, den1d_s are defined in cell 11
# np.interp requires ascending xp; neg_z_vel / neg_z_den are [0, 5000, ...] ascending

vs_ref_lin  = np.interp(neg_z, neg_z_vel, vel1d_s['vs'].values)       # km/s
den_ref_lin = np.interp(neg_z, neg_z_den, den1d_s['den_si'].values)   # kg/m³
mu_ref_at_dofs_lin = den_ref_lin * (vs_ref_lin * 1e3)**2 / 1e9        # GPa

dlnmu_orig_lin = (dof_mu_gpa - mu_ref_at_dofs_lin) / mu_ref_at_dofs_lin

print(f'μ_ref step   range (hull): {mu_ref_at_dofs[inside_3d].min():.2f} – {mu_ref_at_dofs[inside_3d].max():.2f} GPa')
print(f'μ_ref linear range (hull): {mu_ref_at_dofs_lin[inside_3d].min():.2f} – {mu_ref_at_dofs_lin[inside_3d].max():.2f} GPa')
print()
print(f'δln(μ) step   (hull): mean={dlnmu_orig[inside_3d].mean():.4f}  '
      f'range=[{dlnmu_orig[inside_3d].min():.3f}, {dlnmu_orig[inside_3d].max():.3f}]')
print(f'δln(μ) linear (hull): mean={dlnmu_orig_lin[inside_3d].mean():.4f}  '
      f'range=[{dlnmu_orig_lin[inside_3d].min():.3f}, {dlnmu_orig_lin[inside_3d].max():.3f}]')


In [ ]:
# ── 3-Panel: linear-ref δln(μ) unscaled / scaled ×4 / absolute μ ───────────
depth_max = 100  # km
d_hull    = depth_dof_km[inside_3d]

# Scaled δln and recovered absolute μ (linear-interp reference)
dlnmu_scaled_lin = dlnmu_orig_lin * 4
mu_abs_lin       = mu_ref_at_dofs_lin * (1.0 + dlnmu_scaled_lin)   # GPa

fig, axes = plt.subplots(1, 3, figsize=(13, 6), sharey=True)
fig.subplots_adjust(wspace=0.08)
fig.suptitle('Linear-interpolation 1D reference  (hull DOFs, CG2, unscaled input)',
             fontsize=11)

# ── (a) δln(μ) unscaled: step-function vs linear reference ──────────────────
ax = axes[0]
# ax.scatter(dlnmu_orig[inside_3d],     d_hull, color='steelblue', s=2, alpha=0.2,
#            linewidths=0, label='Step-function ref')
ax.scatter(dlnmu_orig_lin[inside_3d], d_hull, color='tomato',    s=2, alpha=0.2,
           linewidths=0, label='Linear-interp ref')
ax.axvline(0, color='k', lw=1.5)
ax.set_xlabel(r'$\delta\ln(\mu)$ unscaled', fontsize=11)
ax.set_ylabel('Depth (km)', fontsize=12)
ax.set_ylim(depth_max, 0)
ax.grid(True, ls='--', alpha=0.4)
ax.set_title('(a) δln(μ) unscaled', fontsize=10)
ax.legend(fontsize=8, markerscale=4)

# ── (b) δln(μ) scaled ×4 (linear ref) ──────────────────────────────────────
ax = axes[1]
ax.scatter(dlnmu_scaled_lin[inside_3d], d_hull, color='tomato', s=2, alpha=0.2,
           linewidths=0)
ax.axvline(0, color='k', lw=1.5)
ax.set_xlabel(r'$\delta\ln(\mu)$ × 4', fontsize=11)
ax.set_ylim(depth_max, 0)
ax.grid(True, ls='--', alpha=0.4)
ax.set_title('(b) δln(μ) scaled ×4', fontsize=10)

# ── (c) Absolute μ: linear-ref scaled vs saved step-ref scaled ──────────────
ax = axes[2]
ax.scatter(dof_mu_gpa_scaled[inside_3d], d_hull, color='steelblue', s=2, alpha=0.15,
           linewidths=0, label='Step-ref ×4 (saved)')
ax.scatter(mu_abs_lin[inside_3d],        d_hull, color='tomato',    s=2, alpha=0.2,
           linewidths=0, label='Linear-ref ×4')
ax.set_xlabel(r'$\mu$ (GPa)', fontsize=11)
ax.set_ylim(depth_max, 0)
ax.grid(True, ls='--', alpha=0.4)
ax.set_title('(c) Absolute μ (scaled ×4)', fontsize=10)
ax.legend(fontsize=8, markerscale=4)

plt.tight_layout()
if savefig:
    fig.savefig(veldir + 'plt_linref_dln_3panel.pdf', bbox_inches='tight')
    print('Saved: plt_linref_dln_3panel.pdf')
plt.show()


In [ ]:
# ── 4-Panel 2×2 Plot: velocity, density, tabulated μ, mesh-projected μ ────────
fig, axes = plt.subplots(2, 2, figsize=(8, 10), sharey=True)
fig.subplots_adjust(wspace=0.08, hspace=0.3)

panel_labels = ["(a)", "(b)", "(c)", "(d)"]

depth_max = 100  # km

# ── Panel [0,0]: Vp, Vs, and Vp/Vs ──────────────────────────────────────────
ax = axes[0, 0]
h1 = ax.scatter(vel3d['vp'],          depth_3d_km, color='steelblue', s=10, alpha=0.5,
                linewidths=0, zorder=1, label='Vp (3D)')
h2 = ax.scatter(vel3d['vs'],          depth_3d_km, color='salmon',    s=10, alpha=0.5,
                linewidths=0, zorder=1, label='Vs (3D)')
h3 = ax.scatter(vel3d['vp_vs_ratio'], depth_3d_km, color='gray',      s=10, alpha=0.5,
                linewidths=0, zorder=1, label='Vp/Vs (3D)')
xvp, yvp = layer_xy(depth_vel_km.values, vel1d['vp'].values)
xvs, yvs = layer_xy(depth_vel_km.values, vel1d['vs'].values)
xvr, yvr = layer_xy(depth_vel_km.values, vel1d['vp_vs_ratio'].values)
h4, = ax.plot(xvp, yvp, color='navy',    lw=2, zorder=2, label='Vp (1D)')
h5, = ax.plot(xvs, yvs, color='darkred', lw=2, zorder=2, label='Vs (1D)')
h6, = ax.plot(xvr, yvr, color='black',   lw=2, zorder=2, label='Vp/Vs (1D)')
ax.set_xlabel('Velocity (km/s) or Vp/Vs ratio', fontsize=11)
ax.set_ylabel('Depth (km)', fontsize=12)
ax.set_xlim(1, 9)
ax.set_ylim(depth_max, 0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(0.5))
ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(5))
ax.grid(True, which='major', ls='--', alpha=0.4)
legend_xy = (0.5, 0.33)
ax.legend(handles=[h4, h5, h6, h1, h2, h3], fontsize=9,
          bbox_to_anchor=legend_xy, loc='upper left',
          ncol=1, handlelength=1.2, framealpha=0.9, edgecolor='0.7')
ax.text(0.02, 0.94, panel_labels[0], transform=ax.transAxes, fontsize=12, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='left', va='bottom')

# ── Panel [0,1]: Density ──────────────────────────────────────────────────────
ax = axes[0, 1]
xd, yd = layer_xy(depth_den_km.values, den1d['den'].values)
ax.plot(xd, yd, color='k', lw=2, label='ρ (1D)')
ax.set_xlabel('Density ρ (g/cm³)', fontsize=11)
ax.set_xlim(2.5, 3.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(0.5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(0.1))
ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(5))
ax.legend(fontsize=9, loc='lower left')
ax.grid(True, which='major', ls='--', alpha=0.4)
ax.text(0.02, 0.94, panel_labels[1], transform=ax.transAxes, fontsize=12, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='left', va='bottom')

# ── Panel [1,0]: Shear modulus from tabulated 3D model ───────────────────────
ax = axes[1, 0]
h1 = ax.scatter(mu_gpa_3d, depth_3d_km, color='gray', s=10, alpha=0.5,
           linewidths=0, zorder=1, label='μ (tabulated 3D)')
xmu, ymu = layer_xy(depth_vel_km.values, mu_gpa_1d.values)
h2, = ax.plot(xmu, ymu, color='k', lw=2, zorder=2, label='μ (1D)')
ax.set_xlabel('Shear modulus μ (GPa)', fontsize=11)
ax.set_ylabel('Depth (km)', fontsize=12)
ax.set_xlim(0, 100)
ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(5))
ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(5))
ax.legend(handles=[h2, h1], fontsize=9, loc='lower left')
ax.grid(True, which='major', ls='--', alpha=0.4)
ax.set_title('Tabulated shear-modulus model', fontsize=11)
ax.text(0.02, 0.94, panel_labels[2], transform=ax.transAxes, fontsize=12, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='left', va='bottom')

# ── Panel [1,1]: Mesh-projected shear modulus (CG2 DOFs) ─────────────────────
ax = axes[1, 1]
# h1 = ax.scatter(dof_mu_gpa, depth_dof_km, color='gray', s=1, alpha=0.1,
#            linewidths=0, zorder=1, label='μ (mesh-projected, CG2)')
h1 = ax.scatter(dof_mu_gpa[inside_3d], depth_dof_km[inside_3d], color='gray', s=10, alpha=0.5,
           linewidths=0, zorder=1, label='μ (3D hull region)')
xmu, ymu = layer_xy(depth_vel_km.values, mu_gpa_1d.values)
h2, = ax.plot(xmu, ymu, color='k', lw=2, zorder=2, label='μ (1D)')
ax.set_xlabel('Shear modulus μ (GPa)', fontsize=11)
ax.set_xlim(0, 100)
ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(5))
ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(5))
ax.legend(handles=[h2, h1], fontsize=9, loc='lower left')
ax.grid(True, which='major', ls='--', alpha=0.4)
ax.set_title('Mesh-projected shear-modulus (CG2 DOFs)', fontsize=11)
ax.text(0.02, 0.94, panel_labels[3], transform=ax.transAxes, fontsize=12, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='left', va='bottom')

fig.suptitle('Reference model of DeShon et al. (2006)', fontsize=14)
plt.tight_layout()
# plt.savefig('../DeShon_2006GJI/plt_1Dmodel_profile_4panel.pdf', bbox_inches='tight')
plt.savefig('../DeShon_2006GJI/plt_1Dmodel_profile_4panel.png', bbox_inches='tight', dpi=600)
plt.show()